In [1]:
import torch
import sys
import gc
print(sys.version)
print(f"PyTorch Version: {torch.__version__}")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(torch.cuda.get_device_name(0))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
print(torch.cuda.is_bf16_supported())

3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
PyTorch Version: 2.12.0+cu126
True
1
CUDA Version: 12.6
NVIDIA GeForce RTX 4080 Laptop GPU
True


# LLM Reliability Case Study - Qwen2.5-0.5B-Instruct

#### **Challenge**
Large Language Models generate probabilistic outputs that cannot always be trusted in operational environments, leading to token drift, logical inconsistencies, and unverified execution paths.

#### **Approach**
The experiment implemented an inference-time verification pipeline incorporating:

* **Multi-Sample Response Generation:** Branching generations into $N$ parallel trajectories to explore the model's latent solution space.
* **Cross-Candidate Semantic Evaluation:** Programmatically checking alignment across different reasoning outputs.
* **Consensus Estimation Across Outputs:** Treating confidence as an emergent property of sampling density rather than a scalar token probability.
* **Contradiction Detection:** Analyzing reasoning paths side-by-side to catch logic decay or divergent facts early.
* **Dynamic Uncertainty Scoring:** Flagging unstable distributions across candidate outputs to dynamically calculate system risk.
* **Ground-Truth Validation:** Injecting rigid constraints to check generations against known, deterministic facts.
* **Human-in-the-Loop Escalation:** Safely routing execution to a manual gatekeeper if the consensus metric drops below the stability threshold.

In [2]:
import torch
import numpy as np
import re
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder

# Setup Models
baseline_model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(baseline_model_name, trust_remote_code=True)
baseline_model = AutoModelForCausalLM.from_pretrained(baseline_model_name, device_map="auto")
device = baseline_model.device

# Load Cross-Encoder on CPU to preserve precious GPU VRAM
relevance_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", device="cpu")
from sentence_transformers import SentenceTransformer

# Load the compact bi-encoder on CPU to handle the dense semantic consensus axes
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")

prompts = [
    "What is AI?",
    "Tell me something interesting about Albert Einstein.",
    "Tell me something about Large Language Models.",
    "What is geometry? Explain it step by step.",
    "Explain the concept of entropy in simple terms.",
    "Tell me something about Jean Baudrillard.",
    "Who was David Hilbert?",
    "Give me three facts about London.",
    "Tell me something about love.",
    "Why did Albert Einstein invent the internet in 1969?"
]

# HARD FILTERS
def fatal_filter(samples):
    """
    Only removes genuinely unusable generations.
    Everything else is preserved for ranking.
    """
    passed_samples = []

    for s in samples:
        text = s["text"].strip()

        # FATAL: empty or trivial output
        if len(text) < 10:
            continue

        # FATAL: invalid or corrupted perplexity
        if s["ppl"] is None or s["ppl"] > 1e6:
            continue

        # FATAL: tokenization / byte explosion
        char_len = len(text)
        token_len = len(s.get("tokens", []))
        if token_len == 0 or (char_len / max(token_len, 1)) < 1.5:
            continue

        # FATAL: extreme repetition collapse
        tokens = text.split()
        if len(tokens) >= 20:
            freq = Counter(tokens)
            top_ratio = freq.most_common(1)[0][1] / len(tokens)
            if top_ratio > 0.70:
                continue

        passed_samples.append(s)

    return passed_samples


import random
import numpy as np
import torch

def generate_n(model, tokenizer, inputs, prompt_len, N=5, temp=0.4, top_p=0.9):
    import random
    import numpy as np
    import torch

    samples = []

    # generate independent seeds
    seeds = [random.randint(0, 2**32 - 1) for _ in range(N)]

    for seed in seeds:
        # isolate randomness
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=500,
                do_sample=True,
                temperature=temp,
                top_p=top_p,
                num_return_sequences=1,
            )

        # decode only assistant continuation
        gen_tokens = output[0][prompt_len:]
        text = tokenizer.decode(gen_tokens, skip_special_tokens=True)

        # compute conditional NLL on generated sequence only
        with torch.no_grad():
            outputs = model(output)
            logits = outputs.logits

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = output[:, 1:].contiguous()

            # only evaluate generated region
            gen_shift_start = prompt_len - 1
            gen_logits = shift_logits[0, gen_shift_start:]
            gen_labels = shift_labels[0, gen_shift_start:]

            loss = torch.nn.functional.cross_entropy(
                gen_logits.view(-1, gen_logits.size(-1)),
                gen_labels.view(-1),
                reduction="mean"
            )

            nll = float(loss.item())
            ppl = float(torch.exp(loss).item())

        samples.append({
            "text": text,
            "ppl": ppl,
            "nll": nll,
            "seed_used": seed,
            "tokens": gen_tokens
        })

    return samples

C:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7567.10it/s]


1. We use a test suite of 10 prompts. The final prompt ("Why did Albert Einstein invent the internet in 1969?") is intentionally nasty. It injects a completely false premise designed to provoke the LLM into a hallucination, letting us test how well the downstream pipeline catches a model under pressure
2. generate_n - While it is tempting to generate the $N$ responses in a single batch using standard engine arguments, batching often shares a random seed state that reduces output variety. We took special care to loop generations sequentially and isolate seeds for each pass, forcing the model to explore a wider variety of generated outputs
3. This filter does not judge the quality or meaning of the variants. Its sole purpose is to quickly reject clear rubbish - like empty text, extreme repetition loops, or broken decoding states while preserving all other responses to look at later

## Optimization Plane
Linguistic & NLI Pipeline Foundations
Before running the evaluation loop, this cell initializes the dual-engine scoring architecture (spaCy + DeBERTa-v3) required to compute the trajectory weights.

1. Why spaCy (en_core_web_sm)?
We use spaCy for deterministic grammatical verification. Instead of relying on an LLM to guess if an output is structured correctly, spaCy performs shallow NLP parsing (tokenization, sentence segmentation, and part-of-speech tagging). This allows us to algorithmically flag structural collapse, such as truncated sentences or trailing clauses, before expensive semantic scoring begins.

2. Why DeBERTa-v3-base-mnli-fever-anli for NLI?
Natural Language Inference (NLI) is our engine for factuality and logic validation. This specific DeBERTa-v3 model is fine-tuned on three major benchmarks (MNLI, FEVER, and ANLI), making it exceptionally robust at detecting zero-shot semantic relationships between a prompt's implicit assumptions and the generated outputs. It classifies candidate trajectories into three explicit states: Entailment (logical follow-through), Neutral, or Contradiction (hallucination/drift).

3. Analyzing branch_disagreement
The pipeline evaluates divergence across the parallel streams by measuring how much candidate outputs conflict with each other semantically. High semantic disagreement indicates that the 0.5B model's latent space is highly unstable for that specific prompt, serving as a signature that the model is guessing rather than converging.

4. The Need for dynamic_weights
Not all prompts are created equal. A factual query requires strict NLI constraints, while a creative query requires grammatical fluency. dynamic_weights adjust the scoring calculus on the fly based on the prompt's profile—balancing the trade-offs between strict factual alignment, vocabulary diversity, and structural formatting requirements.

5. Computing contradiction_penalties
Contradiction penalties are applied as a heavy mathematical down-regulation (negative energy multiplier) on any candidate trajectory where the NLI engine registers a high probability of a CONTRADICTION flag against known ground truths or premises. If a trajectory introduces a factual conflict, its score is aggressively tanked early, preventing it from winning the selection process.

In [3]:
# ===========================================================================
# ENERGY-BASED MULTI-OBJECTIVE TRAJECTORY RANKING FUNCTIONAL
# ===========================================================================
import re
import random
import spacy
import numpy as np
import torch
from collections import Counter
from transformers import pipeline

print("Initializing Advanced Linguistic and NLI Pipelines...")

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

nli_pipeline = pipeline(
    "text-classification",
    model="MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli",
    device=0 if torch.cuda.is_available() else -1
)

def ppl_score(sample):
    """
    Axis 1: Heuristic Fluency Mapping via Sequence-Grounded Log-Likelihood.
    Fixes the upstream mean-reduction by extracting token count T and calculating 
    true total NLL and length-normalized PPL.
    """
    # 1. Dynamically extract true token count T from the tensor list
    tokens = sample.get("tokens")
    t = len(tokens) if tokens is not None else 0
    
    if t == 0:
        return 0.0

    # 2. Re-compute the true mathematical metrics requested by the group
    # Upstream 'nll' is currently mean(losses). Convert to sum(losses)
    upstream_nll_is_mean = sample.get("nll")
    if upstream_nll_is_mean is not None:
        true_sum_nll = upstream_nll_is_mean * t
        true_ppl = np.exp(true_sum_nll / t)  # Exp(sum_nll / T)
    else:
        true_ppl = sample.get("ppl")
        
    if true_ppl is None or true_ppl <= 0 or np.isnan(true_ppl) or np.isinf(true_ppl):
        return 0.0
    
    # Non-linear log compression to map smoothly onto the [0, 1] manifold
    return 1.0 / (1.0 + np.log1p(true_ppl))

def semantic_consensus_scores(samples):
    """
    Axis 3: Geometric Consensus Path Calibration.
    Calculates embedding convergence center, completely un-biased by 
    self-referential matrix diagonals.
    """
    if not samples:
        return np.array([]), np.array([])
        
    texts = [s["text"] for s in samples]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True)
    
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized_embeddings = embeddings / (norms + 1e-8)
    similarity_matrix = np.dot(normalized_embeddings, normalized_embeddings.T)
    
    # Remove self-similarity bias
    np.fill_diagonal(similarity_matrix, 0.0)
    
    if len(samples) > 1:
        scores = similarity_matrix.sum(axis=1) / (len(samples) - 1)
    else:
        scores = np.array([1.0])
        
    return scores, similarity_matrix

def named_entity_density(doc):
    """
    Axis 4: Structural Specificity Heuristic.
    Measures density of core factual markers using pre-cached token streams.
    """
    factual_ents = [ent for ent in doc.ents if ent.label_ in {
        "DATE", "TIME", "GPE", "LOC", "ORG", "PERSON", "CARDINAL", "QUANTITY", "ORDINAL", "PERCENT"
    }]
    word_count = max(len(doc.text.split()), 1)
    
    raw_density = len(factual_ents) / word_count
    return min(raw_density, 0.3)

def analyze_branch_disagreement(samples, similarity_matrix):
    """
    Epistemic Instability Engine.
    Monitors global cohort chaos to dynamically re-allocate feature weights.
    """
    if len(samples) <= 1:
        return {"semantic_drift": 0.0, "avg_numeric_clashes": 0.0, "avg_entity_drift": 0.0}
        
    upper_tri_indices = np.triu_indices_from(similarity_matrix, k=1)
    pairwise_distances = 1.0 - similarity_matrix[upper_tri_indices]
    semantic_drift = float(np.mean(pairwise_distances)) if len(pairwise_distances) > 0 else 0.0
    
    sample_entities = []
    sample_numbers = []
    for s in samples:
        text_lower = s["text"].lower()
        ents = set(re.findall(r'\b[a-z]{4,}\b', text_lower)) 
        nums = set(re.findall(r'\b\d+\b', text_lower))
        sample_entities.append(ents)
        sample_numbers.append(nums)
        
    total_num_clashes = 0
    total_ent_drift = 0
    pairs_checked = 0
    
    for i in range(len(samples)):
        for j in range(i + 1, len(samples)):
            total_num_clashes += len(sample_numbers[i].symmetric_difference(sample_numbers[j]))
            total_ent_drift += len(sample_entities[i].symmetric_difference(sample_entities[j]))
            pairs_checked += 1
            
    return {
        "semantic_drift": semantic_drift,
        "avg_numeric_clashes": total_num_clashes / pairs_checked if pairs_checked > 0 else 0.0,
        "avg_entity_drift": total_ent_drift / pairs_checked if pairs_checked > 0 else 0.0
    }

def get_dynamic_weights(prompt: str) -> dict:
    p_lower = prompt.lower()
    if any(w in p_lower for w in ["facts", "who was", "where is", "list", "date", "born"]):
        return {
            "intent_tag": "📊 Fact Retrieval Mode",
            "weights": {"calibrated_relevance": 0.45, "fluency_ppl": 0.10, "consensus_score": 0.25, "contradiction": 0.20}
        }
    if any(w in p_lower for w in ["love", "feel", "creative", "opinion", "art", "why"]):
        return {
            "intent_tag": "✨ Open-Ended Conceptual Mode",
            "weights": {"calibrated_relevance": 0.40, "fluency_ppl": 0.30, "consensus_score": 0.20, "contradiction": 0.10}
        }
    if any(w in p_lower for w in ["explain", "simply", "step by step", "simple terms", "how to"]):
        return {
            "intent_tag": "🏫 Pedagogical / Explanatory Mode",
            "weights": {"calibrated_relevance": 0.40, "fluency_ppl": 0.10, "consensus_score": 0.30, "contradiction": 0.20}
        }
    return {
        "intent_tag": "⚖️ General Balanced Mode",
        "weights": {"calibrated_relevance": 0.40, "fluency_ppl": 0.10, "consensus_score": 0.30, "contradiction": 0.20}
    }

def completion_penalty(text):
    text = text.strip()
    if not text: return 1.0
    return 1.0 if text[-1] in ('.', '!', '?', ')', ']', '}', '"', "'", '”', '’') else 0.95

def repetition_penalty(text):
    tokens = text.split()
    if len(tokens) < 20: return 1.0
    top_ratio = Counter(tokens).most_common(1)[0][1] / len(tokens)
    if top_ratio < 0.35: return 1.0
    elif top_ratio < 0.50: return 0.95
    elif top_ratio < 0.70: return 0.80
    else: return 0.50

def compute_contradiction_penalties(samples, docs=None):
    """
    Axis 5: Asymmetric Logical Deviation Classifier (Optimized to O(NC)).
    """
    if len(samples) <= 1:
        return [0.0] * len(samples)

    if docs is None:
        docs = [nlp(s["text"]) for s in samples]

    penalties = []
    claims_per_trajectory = []
    
    for doc in docs:
        sents = [s.text.strip() for s in doc.sents if len(s.text.split()) > 4]
        sents = sorted(sents, key=len, reverse=True)[:3] 
        claims_per_trajectory.append(sents)

    for i, claims_i in enumerate(claims_per_trajectory):
        total_contradictions = 0.0
        comparisons = 0

        for j, claims_j in enumerate(claims_per_trajectory):
            if i == j or not claims_i or not claims_j:
                continue

            pairs = [(c1, c2) for c1 in claims_i for c2 in claims_j]
            sampled_pairs = random.sample(pairs, min(3, len(pairs)))

            for c1, c2 in sampled_pairs:
                try:
                    res = nli_pipeline(c1, text_pair=c2)[0]
                    if res.get("label", "").lower() == "contradiction":
                        total_contradictions += res.get("score", 0.0)
                    comparisons += 1
                except Exception:
                    continue

        penalties.append(total_contradictions / max(comparisons, 1))
        
    return penalties

def calculate_dynamic_weights(drift_score, base_weights):
    fluency_w = base_weights.get("fluency_ppl", 0.10)
    relevance_w = base_weights.get("calibrated_relevance", 0.40)
    
    if drift_score < 0.15:
        contradiction_w = 0.10
        consensus_w = 0.40
    elif drift_score < 0.30:
        contradiction_w = 0.25
        consensus_w = 0.25
    else:
        contradiction_w = 0.45
        consensus_w = 0.05  
        
    total = fluency_w + relevance_w + consensus_w + contradiction_w
    return {
        "fluency": fluency_w / total,
        "relevance": relevance_w / total,
        "consensus": consensus_w / total,
        "contradiction": contradiction_w / total
    }

# Baseline configuration profile mappings
INTENT_WEIGHTS = {
    "📊 Fact Retrieval Mode": {"calibrated_relevance": 0.45, "fluency_ppl": 0.10, "consensus_score": 0.25, "contradiction": 0.20},
    "⚖️ General Balanced Mode": {"calibrated_relevance": 0.40, "fluency_ppl": 0.10, "consensus_score": 0.30, "contradiction": 0.20},
    "✨ Open-Ended Conceptual Mode": {"calibrated_relevance": 0.40, "fluency_ppl": 0.30, "consensus_score": 0.20, "contradiction": 0.10},
    "🏫 Pedagogical / Explanatory Mode": {"calibrated_relevance": 0.40, "fluency_ppl": 0.10, "consensus_score": 0.30, "contradiction": 0.20}
}

def rank_samples_cross_encoder(prompt, samples):
    """
    Multi-Objective Trajectory Ranking Engine.
    """
    if not samples:
        return []

    routing_profile = get_dynamic_weights(prompt)
    docs = [nlp(s["text"]) for s in samples]

    raw_consistencies, similarity_matrix = semantic_consensus_scores(samples)
    raw_densities = [named_entity_density(doc) for doc in docs]
    contradiction_penalties = compute_contradiction_penalties(samples, docs)
    
    instability_diagnostics = analyze_branch_disagreement(samples, similarity_matrix)

    min_cons, max_cons = min(raw_consistencies), max(raw_consistencies)
    min_dens, max_dens = min(raw_densities), max(raw_densities)

    ranked = []

    for i, s in enumerate(samples):
        text = s["text"]
        doc = docs[i]

        completion_w = completion_penalty(text)
        repetition_w = repetition_penalty(text)
        fluency = ppl_score(s)

        valid_claims = [sent.text.strip() for sent in doc.sents if len(sent.text.split()) > 4]
        if not valid_claims:
            calibrated_relevance = 0.5
            blended_logit = 0.0
        else:
            sentence_pairs = [(prompt, sent) for sent in valid_claims[:5]]
            sentence_logits = [float(l) for l in relevance_model.predict(sentence_pairs)]
            mean_logit = np.mean(sentence_logits)
            low_tail = np.percentile(sentence_logits, 20)
            blended_logit = (0.7 * mean_logit) + (0.3 * low_tail)
            calibrated_relevance = 1.0 / (1.0 + np.exp(-0.2 * (blended_logit - (-2.0))))

        calibrated_consensus = 1.0 if abs(max_cons - min_cons) < 1e-8 else (raw_consistencies[i] - min_cons) / (max_cons - min_cons + 1e-8)
        calibrated_density = 1.0 if abs(max_dens - min_dens) < 1e-8 else (raw_densities[i] - min_dens) / (max_dens - min_dens + 1e-8)

        base_weights = routing_profile["weights"]
        adjusted_w = calculate_dynamic_weights(instability_diagnostics["semantic_drift"], base_weights)

        final_score = (
            (adjusted_w["relevance"] * calibrated_relevance) +
            (adjusted_w["fluency"] * fluency) +
            (adjusted_w["consensus"] * calibrated_consensus) - 
            (adjusted_w["contradiction"] * contradiction_penalties[i])
        )

        final_score = np.clip(final_score, 0.0, 1.0) * completion_w * repetition_w
        s["entities"] = [ent.text.strip() for ent in doc.ents if ent.label_ in {"PERSON", "ORG", "GPE"}]

        ranked.append({
            "score": float(final_score),
            "intent_tag": routing_profile["intent_tag"],
            "axes": {
                "calibrated_relevance": calibrated_relevance,
                "blended_logit": blended_logit,
                "fluency_ppl": fluency,
                "consensus_score": calibrated_consensus,
                "entity_density": calibrated_density,
                "completion_penalty": completion_w,
                "repetition_penalty": repetition_w,
                "contradiction_penalty": contradiction_penalties[i]
            },
            "meta": {
                "cohort_diagnostics": instability_diagnostics
            },
            "data": s
        })

    ranked.sort(reverse=True, key=lambda x: x["score"])
    return ranked

Initializing Advanced Linguistic and NLI Pipelines...


Loading weights: 100%|█████████| 202/202 [00:00<00:00, 25033.96it/s]


## Stage 2: Matrix Loop

This cell runs the primary evaluation loop across your prompts, scoring and filtering the generated response variations along four distinct axes:

* **Fluency Axis:** Measures token-level language confidence using conditional perplexity metrics directly from the model.
* **Relevance Axis:** Evaluates topical alignment with the user's intent using cross-encoders.
* **Consensus Axis (`calculate_cohort_spread`):** Analyzes the cluster distance between text embeddings to map semantic drift and cluster spread across candidates. High divergence triggers adaptive risk multipliers or a human-in-the-loop (HITL) review loop.
* **Contradiction Axis:** Cross-checks active text variations side-by-side to penalize diverging claims or logic decay early.

The engine uses these axes to assign **dynamic weights** based on the prompt type, sorting the streams to surface a clear **Champion Branch** and a fallback **Contender Branch**.

In [4]:
# ===========================================================================
# HELPER DEPENDENCIES FOR PIPELINE RUNTIME
# ===========================================================================
import numpy as np
import re

def calculate_cohort_spread(embeddings):
    """Calculates spatial variance drift on cohort embeddings."""
    if len(embeddings) <= 1:
        return 0.0, 0.0
    # Normalize embeddings
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized = embeddings / (norms + 1e-8)
    # Distance Matrix = 1 - Cosine Similarity
    similarity_matrix = np.dot(normalized, normalized.T)
    upper_tri = np.triu_indices_from(similarity_matrix, k=1)
    pairwise_distances = 1.0 - similarity_matrix[upper_tri]
    
    semantic_drift = float(np.mean(pairwise_distances)) if len(pairwise_distances) > 0 else 0.0
    cohort_spread = float(np.std(pairwise_distances)) if len(pairwise_distances) > 0 else 0.0
    return semantic_drift, cohort_spread

def classify_volatility(drift_score):
    """Classifies global cohort instability levels."""
    if drift_score < 0.15:
        return "STABLE_CONVERGENCE"
    elif drift_score < 0.30:
        return "MILD_DIVERGENCE"
    elif drift_score < 0.50:
        return "MODERATE_FLUIDITY"
    else:
        return "HIGH_EPISTEMIC_CHAOS"

def advanced_entity_extraction(text):
    """Extracts tracking entities using the pre-cached spaCy stream."""
    doc = nlp(text)
    return [ent.text.strip() for ent in doc.ents if ent.label_ in {"PERSON", "ORG", "GPE", "DATE"}]


# ===========================================================================
# RUNTIME PIPELINE LOOP (STAGE 2 MATRIX EXECUTION)
# ===========================================================================
pipeline_results = {}

for raw_prompt in prompts:
    print("\n" + "="*75)
    print(f"RUNNING INFERENCE SEARCH FOR: '{raw_prompt}'")
    print("="*75)
    
    messages = [{"role": "user", "content": raw_prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(device)
    prompt_len = inputs.input_ids.shape[1]

    # Generate sequential stochastic paths
    raw_candidates = generate_n(
        model=baseline_model, tokenizer=tokenizer, inputs=inputs,
        prompt_len=prompt_len, N=5, temp=0.4, top_p=0.9
    )
    
    # Apply fatal structural constraints safely
    filtered_candidates = fatal_filter(raw_candidates)
    print(f"-> Fatal Filters: {len(filtered_candidates)}/{len(raw_candidates)} passed structural gating.")
    
    if not filtered_candidates:
        print("❌ Status: Terminated. All generated paths failed structural constraints.")
        print("=" * 75 + "\n")
        continue

    # -------------------------------------------------------------------------
    #  Compute Cohort Metrics upfront on raw filtered data
    # -------------------------------------------------------------------------
    # Extract raw texts for processing
    raw_texts = [{"text": c["text"]} for c in filtered_candidates]
    
    # Calculate contradiction penalties on the stable, unsorted list
    contra_penalties = compute_contradiction_penalties(raw_texts)
    
    # Lock the contradiction scores directly into the dictionary objects to eliminate index mismatch risks
    for idx, cand in enumerate(filtered_candidates):
        cand["contradiction_score"] = 1.0 - contra_penalties[idx]

    # Compute true spatial variance drift on raw unsorted embeddings
    cohort_embeddings = embedding_model.encode([c["text"] for c in filtered_candidates], convert_to_numpy=True)
    semantic_drift, cohort_spread = calculate_cohort_spread(cohort_embeddings)
    
    # Calculate Exponential Uncertainty Multiplier 
    uncertainty_multiplier = np.exp(-2.0 * semantic_drift)
    
    # Determine Human-In-The-Loop (HITL) and Severe Penalty triggers
    requires_human_review = semantic_drift > 0.50
    severe_instability_veto = semantic_drift > 0.65
        
    # Execute base ranking engine to extract cross-encoder matrix metrics
    ranked_outputs = rank_samples_cross_encoder(raw_prompt, filtered_candidates)
    
    # Extract active intent configurations
    base_leader = ranked_outputs[0]
    intent_profile = base_leader['intent_tag']
    base_weights = INTENT_WEIGHTS.get(intent_profile, INTENT_WEIGHTS["⚖️ General Balanced Mode"])
    
    volatility_status = classify_volatility(semantic_drift)
    dynamic_specs = calculate_dynamic_weights(semantic_drift, base_weights)
    
    # Print Upgraded Stage 2 Epistemic Stability Telemetry Report
    print("\n" + "="*75)
    print(f"📊 STAGE 2 ENGINE: INTERNAL EPISTEMIC STABILITY REPORT")
    print(f"-> Active Intent Profile     : {intent_profile}")
    print(f"-> Semantic Divergence Drift : {semantic_drift:.4f} (Exponential Multiplier: {uncertainty_multiplier:.4f}x)")
    print(f"-> Cohort Cluster Spread     : {cohort_spread:.4f}")
    print(f"-> Internal Volatility Status: {volatility_status}")
    if requires_human_review:
        print("-> 🛠️  HITL ROUTING FLAG      : TRUE [Requires Human Verification]")
    if severe_instability_veto:
        print("-> 🔴 CRITICAL INSTABILITY   : TRUE [Applying 75% Score Suppression Veto]")
    print("-" * 75)
    print(f"Dynamic Weight Allocation: Fluency({dynamic_specs['fluency']:.2f}) | Relevance({dynamic_specs['relevance']:.2f}) | Consensus({dynamic_specs['consensus']:.2f}) | Contradiction({dynamic_specs['contradiction']:.2f})")
    print("="*75 + "\n")
    
    # 6. Process, re-score, and filter individual branch candidates
    adjusted_ranked_outputs = []
    for candidate in ranked_outputs:
        champ_metrics = candidate["axes"]
        
        # Pull fluency and relevance values out safely
        raw_ppl = champ_metrics['fluency_ppl']
        raw_relevance = champ_metrics['calibrated_relevance']
        raw_consensus = champ_metrics['consensus_score']
        blended_logit = champ_metrics['blended_logit']
        
        # Track clean spaCy NER list inside data payloads
        candidate["data"]["entities"] = advanced_entity_extraction(candidate["data"]["text"])
        
        # The contradiction score safely travels inside the candidate data payload now
        contradiction_score = candidate["data"].get("contradiction_score", 1.0)
        
        # Compute combined recalibration matrix using dynamic specs
        recalibrated_base = (
            (dynamic_specs["fluency"] * raw_ppl) + 
            (dynamic_specs["relevance"] * raw_relevance) + 
            (dynamic_specs["consensus"] * raw_consensus) + 
            (dynamic_specs["contradiction"] * contradiction_score)
        )
        
        # Handle absolute validation vetoes
        if blended_logit < -5.0:
            recalibrated_base *= 0.1
            
        # Apply exponential multiplier mapping
        final_calibrated_score = recalibrated_base * uncertainty_multiplier
        
        # Apply 75% Score Compression if drift breaches safety bounds (>0.65)
        if severe_instability_veto:
            final_calibrated_score *= 0.25
            
        # Store updated metric fields
        candidate["final_calibrated_score"] = final_calibrated_score
        candidate["recalibrated_base"] = recalibrated_base
        candidate["axes"]["contradiction_score"] = contradiction_score
        candidate["requires_human_review"] = requires_human_review
        adjusted_ranked_outputs.append(candidate)
        
    # Re-sort cohort safely by the hardened epistemic metrics
    adjusted_ranked_outputs.sort(reverse=True, key=lambda x: x["final_calibrated_score"])

    # Save cleanly into your global dictionary
    pipeline_results[raw_prompt] = {
        "intent_profile": intent_profile,
        "semantic_drift": semantic_drift,
        "requires_human_review": requires_human_review,
        "severe_instability_veto": severe_instability_veto,
        "candidates": adjusted_ranked_outputs[:3]
    }
    
    # Render Champion Output
    best_candidate = adjusted_ranked_outputs[0]
    champ_metrics = best_candidate["axes"]
    
    print(f"🥇 CHAMPION BRANCH SELECTED (Calibrated Score: {best_candidate['final_calibrated_score']:.4f} | Recalibrated Base: {best_candidate['recalibrated_base']:.4f})")
    print(f"   | Axis 1 (Fluency Weight: {dynamic_specs['fluency']:.2f})       : {champ_metrics['fluency_ppl']:.4f}")
    print(f"   | Axis 2 (Relevance Weight: {dynamic_specs['relevance']:.2f})     : {champ_metrics['calibrated_relevance']:.4f} [Logit: {champ_metrics['blended_logit']:.2f}]")
    print(f"   | Axis 3 (Consensus Weight: {dynamic_specs['consensus']:.2f})     : {champ_metrics['consensus_score']:.4f}")
    print(f"   | Axis 4 (Contradiction Weight: {dynamic_specs['contradiction']:.2f}) : {champ_metrics['contradiction_score']:.4f} (Penalty: {1.0-champ_metrics['contradiction_score']:.4f})")
    print(f"\n--- CHAMPION TEXT ---")
    print(best_candidate["data"]["text"].strip())
    print(f"---------------------\n")
    
    # Render Contender Output
    if len(adjusted_ranked_outputs) > 1:
        contender = adjusted_ranked_outputs[1]
        cont_metrics = contender["axes"]
        score_delta = best_candidate["final_calibrated_score"] - contender["final_calibrated_score"]
        
        print(f"🥈 CONTENDER BRANCH (Calibrated Score: {contender['final_calibrated_score']:.4f} | Margin Delta: -{score_delta:.4f})")
        print(f"   | Axis 1 (Fluency)                : {cont_metrics['fluency_ppl']:.4f}")
        print(f"   | Axis 2 (Relevance)              : {cont_metrics['calibrated_relevance']:.4f} [Logit: {cont_metrics['blended_logit']:.2f}]")
        print(f"   | Axis 3 (Consensus)              : {cont_metrics['consensus_score']:.4f}")
        print(f"   | Axis 4 (Contradiction Consistency): {cont_metrics['contradiction_score']:.4f}")
        print(f"\n--- CONTENDER TEXT ---")
        print(contender["data"]["text"].strip())
        print(f"---------------------\n")
        
    print("=" * 75 + "\n")


RUNNING INFERENCE SEARCH FOR: 'What is AI?'
-> Fatal Filters: 5/5 passed structural gating.


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



📊 STAGE 2 ENGINE: INTERNAL EPISTEMIC STABILITY REPORT
-> Active Intent Profile     : ⚖️ General Balanced Mode
-> Semantic Divergence Drift : 0.1071 (Exponential Multiplier: 0.8072x)
-> Cohort Cluster Spread     : 0.0275
-> Internal Volatility Status: STABLE_CONVERGENCE
---------------------------------------------------------------------------
Dynamic Weight Allocation: Fluency(0.10) | Relevance(0.40) | Consensus(0.40) | Contradiction(0.10)

🥇 CHAMPION BRANCH SELECTED (Calibrated Score: 0.6272 | Recalibrated Base: 0.7769)
   | Axis 1 (Fluency Weight: 0.10)       : 0.4893
   | Axis 2 (Relevance Weight: 0.40)     : 0.5700 [Logit: -0.59]
   | Axis 3 (Consensus Weight: 0.40)     : 1.0000
   | Axis 4 (Contradiction Weight: 0.10) : 1.0000 (Penalty: 0.0000)

--- CHAMPION TEXT ---
AI stands for "Artificial Intelligence," which refers to the simulation of human intelligence processes by machines in tasks that typically require human intelligence, such as visual perception, speech recognition, 

# System Summary & Architectural Insights (Model 2 Actual Log Evaluation)

A deep diagnostic of the actual Stage 2 telemetry for **Model 2 (Qwen2.5-0.5B-Instruct)** exposes the true behavioral profile of an ultra-compact, instruction-tuned model.

The logs reveal that Model 2 is an incredibly eloquent writer. It generates structured, grammatically pristine prose with high internal confidence. However, because its internal knowledge base is highly compressed, it hallucinates core facts with absolute authority.

Stage 2 successfully monitors language quality and structural alignment, but its statistical mathematical matrix treats these highly organized, confident lies as perfect data—frequently awarding them maximum scores on the Consensus and Contradiction axes.

---

## 📈 1. Stability Tracking: Drift vs. Latent Factual Collapse

In your actual run, Model 2's *Semantic Divergence Drift* sits in a tight band between 0.10 and 0.25, keeping the entire system inside `STABLE_CONVERGENCE` or `MILD_DIVERGENCE`. It never triggers a severe volatility alarm, yet the underlying factual data is completely broken.

| Prompt Topic | Drift Score | Volatility Status | Factual Reality in Selected Champion Branch |
| :--- | :--- | :--- | :--- |
| **What is AI?** | 0.1071 | STABLE_CONVERGENCE | Accurate, safe, standard textbook definition. |
| **Who was David Hilbert?** | 0.1553 | MILD_DIVERGENCE | **Severe Historical Fabrication:** Claims Hilbert won the 1927 Nobel Prize in Physics and the 1936 Fields Medal. |
| **What is geometry?** | 0.1738 | MILD_DIVERGENCE | **Mathematical Error:** Defines a plane as "bounded by two parallel lines" and claims quadrilaterals are formed by combining four triangles. |
| **Jean Baudrillard** | 0.2149 | MILD_DIVERGENCE | **Biographical Hallucination:** Invents his full name as "Jean-Baptiste-Benjamin Baudrillard." |
| **Concept of entropy** | 0.2458 | MILD_DIVERGENCE | **Logical Inversion:** Explicitly states that fewer possibilities mean entropy is higher. |

> 💡 **Key Takeaway:** Model 2 does not display epistemic anxiety by fracturing its outputs when it doesn't know something. Instead, its stochastic paths remain highly stable because it relies on smooth sentence templates. A low drift score here merely tracks structural predictability, completely masking catastrophic factual collapse.

---

## 👥 2. The Consensus & Contradiction Loophole (The Fluent Echo Chamber)

The absolute proof of Stage 2's vulnerability as a "truth selector" is laid bare in the *David Hilbert* and *Entropy* runs. Look at how the Consensus (Axis 3) and Contradiction (Axis 4) metrics react to blatant misinformation:

### 🔹 The David Hilbert Run
* **The Lies:** The Champion text confidently asserts that Hilbert won the Nobel Prize in Physics in 1927 and the Fields Medal in 1936 (neither of which is true).
* **The Telemetry Response:** Axis 3 (Consensus) scores a high **0.7892**, and Axis 4 (Contradiction) scores a perfect **1.0000** (Penalty: 0.0000).
* **Why it happened:** Because the small 0.5B model lacks deep historical granularity, its alternative generation seeds pooled together around the same high-probability mathematical associations (Hilbert + famous 20th-century physics/math awards). Because the branches didn't explicitly contradict each other, the NLI layer (DeBERTa) issued zero penalties.

### 🔹 The Entropy Run
* **The Lies:** The model states that removing face cards leaves fewer possibilities, meaning *"the entropy of the remaining cards is higher."* This is a direct inversion of information theory.
* **The Telemetry Response:** Axis 3 flags a perfect **1.0000 Consensus**. The alternative streams all cleanly converged on the exact same inverted logic, completely blinding the matrix.

---

## 🎯 3. The Cross-Encoder Relevance Trap

The Cross-Encoder Relevance Axis (Axis 2) evaluates how well the response structurally mirrors the intent of the prompt. Model 2 maximizes this score by aggressively mimicking structural formats (like step-by-step guides).

* **The Geometry Example:** The user asked for a "step-by-step" explanation. Model 2 responded with a perfectly formatted markdown list (`Step 1`, `Step 2`, `Step 3`... up to `Step 8`). Because the structural alignment was flawless, the framework accepted it as a viable champion, despite Step 1 containing a broken definition of a geometric plane.
* **The Entropy Example:** The Champion text achieved an elite relevance logit of **-0.06** (its highest relevance score across the entire log). The Cross-Encoder heavily rewarded the text because it used classic explanatory formatting (*"In simpler terms:"*, *"For example:"*, Bullet points), completely unaware that the core example was scientifically incorrect.

---

## 📌 Ultimate Conclusion

These actual logs outline the exact operational boundaries of your system before Stage 3:
* **Axis 1 (Fluency)** confirms the text sounds natural.
* **Axis 2 (Relevance)** confirms the text matches the user's requested layout and topic.
* **Axis 3 & 4 (Consensus / Contradiction)** confirm that the internal generation branches are singing in perfect harmony.

When evaluating Qwen-0.5B, Stage 2 functions as an internal coherence auditor rather than a factual verifier. It excels at identifying unstable reasoning trajectories but remains fundamentally vulnerable to consensus hallucinations, where multiple branches confidently converge on the same false latent representation. It proves that a piece of text can be 100% fluent, 100% relevant, and 100% consistent, while being 100% wrong. This makes your upcoming **Stage 3 Factual Audit System** an absolute necessity—it is the only component capable of looking past the beautiful formatting to strike down these high-scoring historical and scientific fabrications.

# Stage 3 Architecture: Asynchronous Factual Audit System & NLI Verification

While Stage 2 acts as a world-class editor—grading structural, semantic, and consensus alignments within the cohort—**Stage 3: GROUND_TRUTH_REGISTRY** serves as the ultimate epistemic anchor. Executed as an asynchronous post-processing validation layer, its single mission is to subject the top-scoring surviving branches to a ruthless factual audit, piercing the veil of high-confidence, fluent hallucinations.

---

## 🎯 1. The Ground Truth Registry & Adversarial Defenses

The system bypasses typical keyword matching in favor of deep semantic parsing by mapping text fragments against a hardcoded verification layer. The registry handles entities through two distinct defensive postures:

### 🔹 Exclusive Precedence Traps (`is_trap: True`)
Designed specifically to neutralize adversarial injections and active user traps (e.g., *"Why did Albert Einstein invent the internet in 1969?"*). These profiles are evaluated with top-priority precedence. If an NLI alignment matches a forbidden claim in a trap, the system bypasses standard profiling and applies catastrophic scoring penalties immediately.

### 🔹 Standard Entity & Concept Profiles (`is_trap: False`)
Traditional factual baselines categorized by type (`entity_factual`, `conceptual`). Rather than analyzing the essay as a monolithic block, Stage 3 leverages a robust regex engine to split candidate text into distinct sentence vectors, evaluating each atomic assertion independently against two strict claim arrays:
* **Required Claims:** Vital pillars or conceptual anchors that the candidate text *must* entail to satisfy completeness.
* **Forbidden Claims:** Core historical falsehoods, logical inversions, or common small-model hallucination patterns that instantly invalidate the text's integrity.

---

## 🧮 2. The Mathematical Scoring Engine

The execution layer uses a zero-tolerance mathematical matrix to strip away credit from beautifully formatted lies. The system calculates three core metrics using Natural Language Inference (NLI) sentence-pair checks:

### 📊 Completeness Score
Measures the coverage of necessary factual pillars. It computes the ratio of matched required claims to total required claims:

$$\text{Completeness} = \frac{|\text{Matched Required Claims}|}{\max(|\text{Required Claims}|, 1)}$$

### 🚨 Integrity Score (Hard Binary Penalty)
Acts as a zero-tolerance tripwire. If an atomic sentence within the response is found to entail any forbidden claim with an NLI confidence threshold of $\ge 0.75$, the integrity multiplier collapses to absolute zero:

$$\text{Integrity} = \begin{cases} 1.0 & \text{if } |\text{Triggered Forbidden Claims}| = 0 \\ 0.0 & \text{if } |\text{Triggered Forbidden Claims}| \ge 1 \end{cases}$$

### 🏆 Combined Factual Truth Score
The final arbiter of truth. Because the metrics are coupled multiplicatively, an uncompromised text scales purely on its informational completeness, whereas a single elegant lie completely zeros out the entire evaluation track:

$$\text{Truth Score} = \text{Completeness} \times \text{Integrity}$$

---

## 📡 3. Runtime Verification & The Epistemic Gap

The execution layer loops over the pre-sorted candidate branches exported from Stage 2, creating a direct, dual-channel comparison between **Internal Confidence** and **Objective Reality**:

In [11]:
import re
import numpy as np

# ---------------------------------------------------------------------------
# UPDATED GROUND TRUTH REGISTRY (Unified Schemas)
# ---------------------------------------------------------------------------
GROUND_TRUTH_REGISTRY = {
    "einstein_internet": {
        "is_trap": True,
        "aliases": ["einstein invent the internet", "einstein internet", "einstein net", "internet in 1969"],
        "type": "correction",
        "required_claims": ["Einstein did not invent the internet", "The internet was not created by Einstein"],
        "forbidden_claims": ["einstein invented the internet", "einstein created the internet", "einstein net"]
    },
    "hilbert": {
        "is_trap": False,
        "aliases": ["david hilbert", "hilbert's"],
        "type": "entity_factual",
        "required_claims": ["David Hilbert was a German mathematician", "David Hilbert died in 1943"],
        "forbidden_claims": ["won the nobel prize", "invented set theory", "born in goettingen", "died in berlin"]
    },
    "einstein": {
        "is_trap": False,
        "aliases": ["albert einstein", "einstein's"],
        "type": "entity_factual",
        "required_claims": ["Albert Einstein was a physicist", "developed relativity"],
        "forbidden_claims": ["father was alfred", "born on april 14", "painter and composer", "wrote music for operas", "university of munich"]
    },
    "baudrillard": {
        "is_trap": False,
        "aliases": ["jean baudrillard", "baudrillard's"],
        "type": "entity_factual",
        "required_claims": ["Jean Baudrillard was a French sociologist", "wrote about simulation and hyperreality"],
        "forbidden_claims": ["grew up in italy", "university of rome", "消费ism", "book in 1980"]
    },
    "ai": {
        "is_trap": False,
        "aliases": ["artificial intelligence", "what is ai"],
        "type": "conceptual",
        "required_claims": ["simulation of human intelligence", "learn from data"], # Fixed: unified 'anchors' to 'required_claims'
        "forbidden_claims": ["conscious like humans", "single algorithm"]
    }
}

# ---------------------------------------------------------------------------
# SENTENCE SPLITTER
# ---------------------------------------------------------------------------
def split_into_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if len(s.strip()) > 5]

# ---------------------------------------------------------------------------
# ROBUST NLI ENTAILMENT CHECKER (Returns boolean and raw score)
# ---------------------------------------------------------------------------
def check_nli(nli_pipeline, premise: str, hypothesis: str, threshold=0.75):
    try:
        res = nli_pipeline(premise, text_pair=hypothesis)[0]
        label = res.get("label", "").lower()
        score = res.get("score", 0.0)
        is_entail = (label == "entailment") and (score >= threshold)
        return is_entail, score
    except Exception:
        return False, 0.0

# ---------------------------------------------------------------------------
# INSTRUMENTED STAGE 3 EVALUATOR
# ---------------------------------------------------------------------------
def run_stage3_evaluator(candidate_text, prompt_string, nli_pipeline, branch_debug_label=""):
    prompt_lower = prompt_string.lower()
    matched_key = None
    
    for key, gt in GROUND_TRUTH_REGISTRY.items():
        aliases = gt.get("aliases", [])
        if any(alias in prompt_lower for alias in aliases):
            matched_key = key
            break

    if matched_key is None:
        return {
            "type": "unverified", "completeness": 0.0, "integrity": 1.0, "truth_score": 0.5,
            "matched_positive": [], "missing_positive": [], "triggered_forbidden": [],
            "note": "No ground truth key matched; Stage 3 skipped."
        }

    gt = GROUND_TRUTH_REGISTRY[matched_key]
    sentences = split_into_sentences(candidate_text)
    required_claims = gt.get("required_claims", [])
    forbidden_claims = gt.get("forbidden_claims", [])

    matched_positive = []
    missing_positive = []
    
    # 1. REQUIRED CLAIM COVERAGE WITH VERBOSE LOGGING
    for claim in required_claims:
        found = False
        for sent in sentences:
            is_entail, nli_score = check_nli(nli_pipeline, sent, claim)
            if is_entail:
                print(f"   [DEBUG ENTAILMENT] Branch: {branch_debug_label}")
                print(f"     └─ Sentence: \"{sent}\"")
                print(f"     └─ Verified Target: \"{claim}\" (Score: {nli_score:.4f})")
                matched_positive.append(claim)
                found = True
                break
        if not found:
            missing_positive.append(claim)

    completeness = len(matched_positive) / max(len(required_claims), 1)

    # 2. FORBIDDEN CLAIM DETECTION WITH REAL DYNAMIC SCORES
    triggered_forbidden = []
    for bad_claim in forbidden_claims:
        for sent in sentences:
            is_entail, nli_score = check_nli(nli_pipeline, sent, bad_claim)
            if is_entail:
                print(f"   [DEBUG VIOLATION] Branch: {branch_debug_label}")
                print(f"     └─ Sentence: \"{sent}\"")
                print(f"     └─ Triggered Forbidden: \"{bad_claim}\" (Score: {nli_score:.4f})")
                triggered_forbidden.append((sent, bad_claim, nli_score))

    integrity = 1.0 if len(triggered_forbidden) == 0 else 0.0
    truth_score = completeness * integrity

    return {
        "type": gt.get("type", "unknown"),
        "completeness": completeness,
        "integrity": integrity,
        "truth_score": truth_score,
        "matched_positive": matched_positive,
        "missing_positive": missing_positive,
        "triggered_forbidden": triggered_forbidden
    }

# ===========================================================================
# RUNTIME AUDIT PIPELINE EXECUTION WITH DETERMINISTIC ORCHESTRATION ROUTING
# ===========================================================================
print("="*75)
print("🎯 DECOUPLED RUNTIME: STAGE 3 POST-GENERATION FACTUAL AUDIT SYSTEM")
print("="*75)

if 'pipeline_results' not in locals() and 'pipeline_results' not in globals():
    print("❌ Error: pipeline_results data dictionary structure cannot be found.")
else:
    for raw_prompt, data in pipeline_results.items():
        candidates = data.get("candidates", [])
        intent_profile = data.get("intent_profile", "⚖️ General Balanced Mode")
        semantic_drift = data.get("semantic_drift", 0.0)
        
        if not candidates:
            continue
            
        uncertainty_multiplier = np.exp(-2.0 * semantic_drift)
        print("\n" + "="*75)
        print(f"📊 HISTORICAL STAGE 2 ENGINE TELEMETRY MATRIX FOR: '{raw_prompt}'")
        print(f"-> Active Intent Profile     : {intent_profile}")
        print(f"-> Semantic Divergence Drift : {semantic_drift:.4f} (Multiplier: {uncertainty_multiplier:.4f}x)")
        print("-" * 75)
        
        best_candidate = candidates[0]
        
        # Define candidate mapping
        loop_candidates = {"🥇 CHAMPION BRANCH": best_candidate}
        if len(candidates) > 1:
            loop_candidates["🥈 CONTENDER BRANCH"] = candidates[1]

        # 1. EVALUATION PHASE: Process all branches and cache reports
        branch_reports = {}
        for branch_name, candidate in loop_candidates.items():
            raw_text = candidate["data"]["text"]
            
            # Run evaluator with real-time debug tracking onto console
            print(f"\n🔍 Auditing {branch_name}...")
            report = run_stage3_evaluator(raw_text, raw_prompt, nli_pipeline, branch_debug_label=branch_name)
            
            # Store report data for our routing gate
            branch_reports[branch_name] = {
                "report": report,
                "text": raw_text,
                "score": candidate.get('final_calibrated_score', 0.0)
            }

        # 2. ORCHESTRATION PHASE: Apply deterministic routing rules
        print("\n" + "═"*75)
        print("🤖 AUTOMATED ORCHESTRATION LAYER DECISION MATRIX")
        print("═"*75)
        
        champ_data = branch_reports["🥇 CHAMPION BRANCH"]
        champ_rep = champ_data["report"]
        
        final_output_stream = ""
        routing_reason = ""
        
        # Check if we have a two-branch system to compare
        if "🥈 CONTENDER BRANCH" in branch_reports:
            cont_data = branch_reports["🥈 CONTENDER BRANCH"]
            cont_rep = cont_data["report"]
            
            # RULE 1: Both pass integrity checks -> Rank based on highest Completeness
            if champ_rep["integrity"] == 1.0 and cont_rep["integrity"] == 1.0:
                if cont_rep["completeness"] > champ_rep["completeness"]:
                    final_output_stream = cont_data["text"]
                    routing_reason = (f"🔄 OVERRIDE ROUTING: Both branches preserved integrity, "
                                      f"but CONTENDER won out on completeness "
                                      f"({cont_rep['completeness']*100:.1f}% vs {champ_rep['completeness']*100:.1f}%).")
                else:
                    final_output_stream = champ_data["text"]
                    routing_reason = "✅ MAINTAIN CHAMPION: Both branches preserved integrity. Champion maintained factual superiority/parity."
            
            # RULE 2: Safe Routing -> One passes, one fails
            elif champ_rep["integrity"] != cont_rep["integrity"]:
                if champ_rep["integrity"] == 1.0:
                    final_output_stream = champ_data["text"]
                    routing_reason = "✅ MAINTAIN CHAMPION: Champion preserved absolute factual integrity while Contender collapsed into hallucination."
                else:
                    final_output_stream = cont_data["text"]
                    routing_reason = "🔄 OVERRIDE ROUTING: Champion failed critical integrity benchmarks! Contender cleanly routed to protect production stream."
            
            # RULE 3: Catastrophic Double-Fail -> Strip contents and return hard fallback message
            elif champ_rep["integrity"] == 0.0 and cont_rep["integrity"] == 0.0:
                final_output_stream = "⚠️ I am unable to verify the historical or factual accuracy of the claims required to fulfill this response."
                routing_reason = "❌ CATASTROPHIC SYSTEM FAILURE: Both generation channels triggered zero-integrity metrics due to severe hallucinations. Hard fallback deployed."
        
        else:
            # Single branch fallback rule block
            if champ_rep["integrity"] == 0.0:
                final_output_stream = "⚠️ I am unable to verify the historical or factual accuracy of the claims required to fulfill this response."
                routing_reason = "❌ SINGLE BRANCH BREAK: Lone champion branch failed critical integrity benchmarks. Hard fallback deployed."
            else:
                final_output_stream = champ_data["text"]
                routing_reason = "✅ PASS-THROUGH: Single branch verification confirmed stable integrity parameters."

        # 3. PRODUCTION STREAM DELIVERY
        print(f"-> Decision Verdict : {routing_reason}")
        print("-" * 75)
        print("🚀 FINAL PRODUCTION STREAM OUTPUT:")
        print("-" * 75)
        print(final_output_stream.strip())
        print("=" * 75 + "\n")

🎯 DECOUPLED RUNTIME: STAGE 3 POST-GENERATION FACTUAL AUDIT SYSTEM

📊 HISTORICAL STAGE 2 ENGINE TELEMETRY MATRIX FOR: 'What is AI?'
-> Active Intent Profile     : ⚖️ General Balanced Mode
-> Semantic Divergence Drift : 0.1071 (Multiplier: 0.8072x)
---------------------------------------------------------------------------

🔍 Auditing 🥇 CHAMPION BRANCH...
   [DEBUG ENTAILMENT] Branch: 🥇 CHAMPION BRANCH
     └─ Sentence: "AI stands for "Artificial Intelligence," which refers to the simulation of human intelligence processes by machines in tasks that typically require human intelligence, such as visual perception, speech recognition, decision-making, and language translation."
     └─ Verified Target: "simulation of human intelligence" (Score: 0.9890)

🔍 Auditing 🥈 CONTENDER BRANCH...
   [DEBUG ENTAILMENT] Branch: 🥈 CONTENDER BRANCH
     └─ Sentence: "AI stands for "Artificial Intelligence," which refers to the simulation of human intelligence processes by machines in tasks that typically 

# Stage 3 Runtime Verification Report
## Epistemic Gap Analysis: Internal Confidence vs Objective Reality

### Overview

Stage 3 successfully demonstrated the necessity of an external factual verification layer operating independently from the Stage 2 ranking matrix.

While Stage 2 effectively selected outputs that were fluent, relevant, and internally consistent, Stage 3 exposed multiple cases where highly-ranked responses contained severe factual inaccuracies.

The experiment validates the central hypothesis of this case study:

> **Statistical confidence is not equivalent to factual correctness.**

---

# 1. Major Architectural Successes

## 1.1 Ground Truth Registry Successfully Detected Severe Hallucinations

The registry-based NLI audit correctly identified high-confidence factual failures that Stage 2 alone could not detect.

### Einstein Internet Trap

**Stage 2 Champion:**
- High fluency
- High relevance
- Low contradiction penalty
- Selected as best response

*Generated claim:*
> "Albert Einstein invented the internet in 1969..."

Stage 3 immediately detected:
- **Forbidden claim:** *"einstein invented the internet"*
- **Forbidden claim:** *"einstein created the internet"*
- **Forbidden claim:** *"einstein net"*

**Result:**
* Integrity Score $\rightarrow$ `0.0`
* Truth Score $\rightarrow$ `0.0`
* **Hard Fallback Activated**

### Interpretation
This demonstrates that:
- Consensus across branches **does not imply truth**.
- Small models can confidently converge on false premises.
- Stage 3 successfully neutralizes adversarial prompt injections.

---

## 1.2 Historical Hallucinations Were Successfully Suppressed

### David Hilbert

**Stage 2 Champion:**
- Consensus $\approx$ High
- Contradiction = Perfect
- Strong relevance

*Generated hallucinations:*
- Hilbert won a Nobel Prize
- Born in Göttingen (forbidden registry entry)

Stage 3 detected:
* **Forbidden:** `✓ won the nobel prize`
* **Forbidden:** `✓ born in goettingen`

**Contender branch:**
* **Detected:** `✓ won the nobel prize`

**Result:**
* Both branches failed integrity checks.
* **System fallback triggered.**

### Interpretation
Stage 3 prevented a highly-ranked historical fabrication from reaching production output.

---

## 1.3 Registry Anchors Improved Branch Selection

### AI Prompt

**Champion:**
* *Matched:* `simulation of human intelligence`
* *Completeness:* `50%`

**Contender:**
* *Matched:* `simulation of human intelligence`, `learn from data`
* *Completeness:* `100%`

**Result:**
* Champion overridden.
* **Contender promoted.**

### Interpretation
Stage 3 is not purely punitive. It can actively improve output quality by selecting branches with better factual coverage.

---

# 2. Epistemic Gap Observations

The experiments revealed a consistent gap between **Stage 2 Internal Confidence** (measuring fluency, prompt alignment, cohort agreement, and internal consistency) versus **Stage 3 Objective Reality** (measuring ground truth coverage, forbidden claim detection, and external factual constraints).

### Epistemic Gap Matrix

| Prompt | Stage 2 Assessment | Stage 3 Assessment |
| :--- | :--- | :--- |
| **AI** | Strong | Improved via completeness override |
| **Einstein (Biographical)** | Strong | Contender isolated / Champion passed |
| **David Hilbert** | Strong | Catastrophic failure (Hard Blocked) |
| **Einstein Internet Trap** | Moderate | Catastrophic failure (Hard Blocked) |
| **Geometry** | Strong | Not evaluated (Passed Through) |
| **Entropy** | Strong | Not evaluated (Passed Through) |
| **Baudrillard** | Strong | Partial correction (Contender Dropped) |

---

# 3. Remaining Limitations Revealed by Stage 3

## 3.1 Registry Coverage Bottleneck
Several prompts bypassed factual auditing entirely because no registry entries existed (e.g., *Geometry, Entropy, London, Love, Large Language Models*). 

Stage 3 defaulted to `"Integrity preserved"`. This means:
> **Absence of evidence is not evidence of correctness.** 

Stage 3 only verifies what it explicitly knows.

## 3.2 NLI False Positives
Some forbidden claim matches were semantically questionable. For example, in the **Baudrillard** prompt, the forbidden token `"消费ism"` was triggered by the English phrase *"...consumerism on society."* 

While technically flagged, this is an artifact of semantic similarity rather than a true hallucination, confirming that **NLI $\neq$ a flawless Fact Checker**. Threshold tuning remains critical.

## 3.3 Partial Contradiction Behaviour
The Einstein Internet Contender produced a correct statement (*"Einstein himself didn't invent the internet."*) while simultaneously claiming *"Einstein is often credited as the inventor..."*. 

Stage 3 correctly assigned a final Integrity of `0` because required claims were satisfied but forbidden claims were triggered. This demonstrates that **a single truthful sentence cannot rescue an otherwise false narrative.**

---

# 4. What Stage 3 Actually Achieved

The system successfully transitioned the pipeline from a Stage 2 mindset of *"Which response appears strongest internally?"* to a Stage 3 mindset of *"Does this response satisfy external factual constraints?"* 

This introduced:
* Claim-level entailment auditing
* Explicit hallucination detection
* Adversarial prompt defense
* Registry-driven factual anchoring
* Production fallback mechanisms
* Champion branch overrides

---

# 5. Key Finding of the Case Study

The complete pipeline demonstrates a fundamental truth of working with LLMs:
* **Fluent $\neq$ Correct**
* **Relevant $\neq$ Correct**
* **Consistent $\neq$ Correct**
* **Consensus $\neq$ Correct**

Only external verification against independent factual constraints can identify stable hallucinations, consensus-driven misinformation, and high-confidence factual collapse. Stage 3 therefore functions as the pipeline's **Epistemic Safety Layer**, bridging the gap between internal model confidence and objective reality.

---

# Final Conclusion

The combined architecture demonstrates a critical property of trusted LLM systems: A model can be simultaneously **fluent, relevant, internally consistent, and highly confident while being completely factually wrong.**

Stage 2 measures **how confidently the model believes itself.** Stage 3 measures **whether reality agrees.** Together, they form a practical reliability framework for small local language models operating in environments where factual integrity is non-negotiable and hallucinations carry high operational risk.
